In [1]:
!pip install --upgrade pydantic>=2.0 openai anthropic google-generativeai together pandas numpy

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.15.1 requires numpy<2.0.0,>=1.23.5, but you have numpy 2.2.6 which is incompatible.
spacy 3.4.4 requires pydantic!=1.8,!=1.8.1,<1.11.0,>=1.7.4, but you have pydantic 2.12.5 which is incompatible.
spacy 3.4.4 requires typer<0.8.0,>=0.3.0, but you have typer 0.19.2 which is incompatible.
scipy 1.9.3 requires numpy<1.26.0,>=1.18.5, but you have numpy 2.2.6 which is incompatible.
deepnote-toolkit 1.1.5 requires numpy<2,>=1.23; python_version == "3.10", but you have numpy 2.2.6 which is incompatible.
deepnote-toolkit 1.1.5 requires pandas<2.2,>=1.2.5; python_version < "3.12", but you have pandas 2.3.3 which is incompatible.

[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
pip install --upgrade pip

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 108.3 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 23.0.1
    Uninstalling pip-23.0.1:
      Successfully uninstalled pip-23.0.1
Note: you may need to restart the kernel to use updated packages.


In [7]:
"""
ABLATION STUDY - USING ORIGINAL STAGE 8 PROMPTS
================================================
Tests which features matter most for LLM predictions by systematically removing them.

This version uses the EXACT prompt format from the original study (Stage 8)
but applies the same ablation methodology from the previous ablation study.

FEATURES:
1. ✓ Checkpoint/resume system (can resume after failures)
2. ✓ Keep-alive thread for long runs (prevents timeout)
3. ✓ Retry logic for API calls with exponential backoff
4. ✓ All same ablation conditions tested
5. ✓ Counterfactual tests (GDP flip, name-data mismatch, willingness flip)
6. ✓ Progress tracking and time estimates
7. ✓ Forbidden string detection (prevents academic citations)
8. ✓ Can fill missing predictions later

USAGE:
1. First run: python ablation_study_original_prompts.py
2. If interrupted: Just run again - will resume from checkpoint
3. To fill missing: Use fill_missing_original.py (created separately)
"""

import os, re, time, json, threading
from datetime import datetime
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# API imports
from openai import OpenAI
import anthropic
import google.generativeai as genai
from together import Together

print("="*80)
print("ABLATION STUDY - ORIGINAL STAGE 8 PROMPTS")
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

# ================================================================
# Keep-alive thread (prevents timeout on long runs)
# ================================================================

def keep_alive():
    """Print periodic messages to keep session alive."""
    while True:
        if time.time() % 300 < 60:  # Every 5 minutes
            print(f"\n⏱️  [{datetime.now().strftime('%H:%M:%S')}] Session alive", flush=True)
        else:
            print(".", end="", flush=True)
        time.sleep(60)

keep_alive_thread = threading.Thread(target=keep_alive, daemon=True)
keep_alive_thread.start()
print("✓ Keep-alive thread started (prevents timeout)\n")

# ================================================================
# Configuration
# ================================================================

CHECKPOINT_FILE = "ablation_original_checkpoint.json"
RAW_RESULTS_FILE = "ablation_original_raw_results.csv"
DATA_FILE = "data_final.csv"

# API Configuration
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
CLAUDE_API_KEY = os.getenv("CLAUDE_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
LLAMA_API_KEY = os.getenv("LLAMA_API_KEY")

# Model configurations
MODELS = {
    "gpt": "gpt-4o-mini",
    "claude": "claude-3-5-haiku-20241022",
    "gemini": "gemini-2.5-flash",  # Google AI Studio: Gemini 2.5 Flash
    "llama": "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8"  # Together AI: Meta's Llama 4 Maverick
}

# System instruction to prevent data leakage and ensure lower-bound performance test
SYSTEM_INSTRUCTION = """You are a prediction assistant making estimates based ONLY on the information provided in this specific prompt.

CRITICAL INSTRUCTIONS:
1. Do NOT cite, reference, or mention ANY research papers, academic studies, surveys, or authors (including but not limited to André et al., Sparkman et al., Leviston et al., Lees et al., or any other researchers)
2. Do NOT use any memorized data, statistics, or percentages from your training about climate change opinions, pluralistic ignorance, or survey results
3. Treat this as a completely NOVEL scenario - ignore any similar studies you may have seen during training
4. Do NOT reference 'research shows', 'studies indicate', 'surveys have found', or similar phrases
5. Base your estimate ONLY on:
   - General reasoning about human psychology and behavior
   - The specific information provided in this prompt
   - First principles about how people form beliefs about others

Your task is to predict what percentage people THINK others believe (second-order belief), not what people actually believe (first-order belief). This is a prediction task requiring general reasoning, not recall of specific research findings.

Respond with ONLY a JSON object containing a single number between 0 and 100 with one decimal place: {"prediction": XX.X}

Do not include any explanation, reasoning, or text - only the JSON."""

# Condition order (INCLUDING NEW COUNTERFACTUAL)
condition_order = [
    'full',                 # Baseline - all features
    'no_econ',             # Remove economic indicators
    'no_religion',         # Remove religion
    'no_demo',             # Remove demographics
    'no_climate',          # Remove climate
    'no_own_willingness',  # Remove personal willingness (CRITICAL)
    'country_only',        # Only country name
    'cf_gdp_flip',         # Counterfactual: flip GDP
    'cf_name_mismatch',    # Counterfactual: mismatch name and data
    'cf_willingness_flip'  # NEW: Counterfactual: flip own willingness
]

# ================================================================
# Helper Functions
# ================================================================

def as_num(x, nd=1):
    """Convert to number with specified decimal places."""
    if pd.isna(x): 
        return None
    try:
        return round(float(x), nd)
    except:
        return None

def as_pct(x):
    """Return percentage (0-100) as float. Accept 0–1 or 0–100."""
    if pd.isna(x): 
        return None
    try:
        v = float(x)
    except:
        return None
    if 0 <= v <= 1:  # share format (0-1)
        return v * 100.0
    return v

def fmt_pct(x):
    """Format as percentage string."""
    v = as_pct(x)
    return f"{v:.1f}%" if v is not None else "N/A"

def fmt_num(x, nd=1):
    """Format as number string."""
    v = as_num(x, nd)
    return f"{v:.{nd}f}" if v is not None else "N/A"

def extract_number_0_100(text):
    """Extract a number between 0-100 from text or JSON."""
    if not isinstance(text, str):
        return None
    
    # Try parsing as JSON first
    try:
        data = json.loads(text)
        if isinstance(data, dict):
            # Look for common keys
            for key in ["prediction", "estimate", "value", "number", "percentage"]:
                if key in data:
                    val = float(data[key])
                    return max(0.0, min(100.0, val))
        elif isinstance(data, (int, float)):
            val = float(data)
            return max(0.0, min(100.0, val))
    except:
        pass
    
    # Fallback: regex extraction
    m = re.search(r"(\d+(?:\.\d+)?)", text)
    if not m:
        return None
    val = float(m.group(1))
    return max(0.0, min(100.0, val))

def contains_forbidden_strings(text):
    """Check if response contains forbidden strings that suggest academic citation."""
    if not isinstance(text, str):
        return False
    
    text_lower = text.lower()
    forbidden = [
        "andré", "andre",  # Author name variations
        "et al", "et. al", "et.al",  # Citation markers
        "doi", "http://", "https://",  # Links/DOIs
        "paper", "study", "research",  # Academic references (be careful with these)
        "published", "journal", "article"  # Publication terms
    ]
    
    return any(term in text_lower for term in forbidden)

# ================================================================
# Load Data
# ================================================================

print("\n1. Loading data...")
if not Path(DATA_FILE).exists():
    print(f"   ❌ Data file not found: {DATA_FILE}")
    print("   Please ensure data_final.csv is in the same directory")
    exit(1)

df = pd.read_csv(DATA_FILE)
print(f"   ✓ Loaded {len(df)} countries")

# Identify column names (handle spelling variations)
OWN_LESS_COL = None
for col_name in ["mean_own_willingness_less", "mean_own_willigness_less"]:
    if col_name in df.columns:
        OWN_LESS_COL = col_name
        break

TEMP_COL = None
for col_name in ["temp_mean", "temp_mean_2010_2019"]:
    if col_name in df.columns:
        TEMP_COL = col_name
        break

print(f"   ✓ Temperature column: {TEMP_COL}")
print(f"   ✓ Willingness_less column: {OWN_LESS_COL}")

# ================================================================
# Original Stage 8 Prompt Builder
# ================================================================

def build_prompt_stage8_original(row):
    """
    Build the ORIGINAL Stage 8 prompt exactly as in the original study.
    This is the baseline "full" condition.
    """
    country = row["countrynew"]
    
    socio_demographics = (
        f"The average age of respondents is {fmt_num(row.get('mean_age'), 1)} years, "
        f"{fmt_pct(row.get('mean_edu'))} of the people have completed a tertiary education, "
        f"and {fmt_pct(row.get('mean_religion'))} say religion is important in daily life."
    )
    
    macro_economic = (
        f"GDP per capita (PPP, 2021) is ${fmt_num(row.get('gdp_capita_2021'), 0)}, "
        f"the top 1% holds {fmt_pct(row.get('top1pct_income'))} of total income, "
        f"and {fmt_pct(row.get('top1pct_wealth'))} of total wealth. "
        f"The Human Development Index (2021) is {fmt_num(row.get('hdi_2021'), 3)}."
    )
    
    if TEMP_COL:
        temperature = f"The average temperature from 2010 to 2019 is {fmt_num(row.get(TEMP_COL), 1)}°C."
    else:
        temperature = "Temperature data are not available."
    
    own_main = fmt_pct(row.get("mean_own_willingness"))
    own_less = fmt_pct(row.get(OWN_LESS_COL)) if OWN_LESS_COL else "N/A"
    
    willingness = (
        f"In this survey, {own_main} of people said they are personally willing to contribute 1% of their income each month, "
        f"and an additional {own_less} would contribute a smaller amount."
    )
    
    ask = (
        f"In a nationally representative survey with a probability-based sample of approximately 1000 residents aged 15 and above in {country}, respondents were asked: "
        f"'Would you be willing to contribute 1% of your household income every month to fight global warming? This would mean that you would contribute 1 for every 100 of this income.' "
        f"Responses: Yes, No, (Don't Know), (Refused). Don't know and refused were coded as missing data. Respondents were then asked how many respondents in {country} they think are willing to contribute at least 1% of their household income every month to fight global warming. "
        f"Responses: between 0% and 100%, (Don't know), (Refused). Based on the country, socio-demographic, macro-economic indicators, temperature data, and the actual willingness data shown above, estimate what respondents in {country} on average thought about how many OTHER respondents in {country} are willing to contribute at least 1% of their household income every month to fight global warming. Note: You are estimating people's BELIEFS about others' willingness, not the actual willingness itself. "
        "Respond with a single number between 0 and 100, with one decimal place."
        
    )
    
    return f"In {country}, {socio_demographics} {macro_economic} {temperature} {willingness} {ask}".replace("..", ".")

# ================================================================
# Ablation Prompt Builders (using original Stage 8 format)
# ================================================================

def build_ablated_prompts(row, all_countries_df):
    """
    Build all ablation versions using the original Stage 8 format.
    Returns a dict with all conditions.
    """
    country = row["countrynew"]
    prompts = {}
    
    # Build reusable components
    socio_full = (
        f"The average age of respondents is {fmt_num(row.get('mean_age'), 1)} years, "
        f"{fmt_pct(row.get('mean_edu'))} of the people have completed a tertiary education, "
        f"and {fmt_pct(row.get('mean_religion'))} say religion is important in daily life."
    )
    
    socio_no_religion = (
        f"The average age of respondents is {fmt_num(row.get('mean_age'), 1)} years, "
        f"and {fmt_pct(row.get('mean_edu'))} of the people have completed a tertiary education."
    )
    
    macro_economic = (
        f"GDP per capita (PPP, 2021) is ${fmt_num(row.get('gdp_capita_2021'), 0)}, "
        f"the top 1% holds {fmt_pct(row.get('top1pct_income'))} of total income, "
        f"and {fmt_pct(row.get('top1pct_wealth'))} of total wealth. "
        f"The Human Development Index (2021) is {fmt_num(row.get('hdi_2021'), 3)}."
    )
    
    gdp_only = f"GDP per capita (PPP, 2021) is ${fmt_num(row.get('gdp_capita_2021'), 0)}."
    
    religion_only = f"{fmt_pct(row.get('mean_religion'))} say religion is important in daily life."
    
    if TEMP_COL:
        temperature = f"The average temperature from 2010 to 2019 is {fmt_num(row.get(TEMP_COL), 1)}°C."
    else:
        temperature = "Temperature data are not available."
    
    own_main = fmt_pct(row.get("mean_own_willingness"))
    own_less = fmt_pct(row.get(OWN_LESS_COL)) if OWN_LESS_COL else "N/A"
    
    willingness = (
        f"In this survey, {own_main} of people said they are personally willing to contribute 1% of their income each month, "
        f"and an additional {own_less} would contribute a smaller amount."
    )
    
    ask_full = (
        f"In a nationally representative survey with a probability-based sample of approximately 1000 residents aged 15 and above in {country}, respondents were asked: "
        f"'Would you be willing to contribute 1% of your household income every month to fight global warming? This would mean that you would contribute 1 for every 100 of this income.' "
        f"Responses: Yes, No, (Don't Know), (Refused). Don't know and refused were coded as missing data. Respondents were then asked how many respondents in {country} they think are willing to contribute at least 1% of their household income every month to fight global warming. "
        f"Responses: between 0% and 100%, (Don't know), (Refused). Based on the country, socio-demographic, macro-economic indicators, temperature data, and the actual willingness data shown above, estimate what respondents in {country} on average thought about how many OTHER respondents in {country} are willing to contribute at least 1% of their household income every month to fight global warming. Note: You are estimating people's BELIEFS about others' willingness, not the actual willingness itself. "
        "Respond with a single number between 0 and 100, with one decimal place."
    )
    
    # FULL PROMPT (baseline)
    prompts['full'] = build_prompt_stage8_original(row)
    
    # ABLATION 1: No Economic Indicators
    prompts['no_econ'] = f"In {country}, {socio_full} {temperature} {willingness} {ask_full}".replace("..", ".")
    
    # ABLATION 2: No Religion
    prompts['no_religion'] = f"In {country}, {socio_no_religion} {macro_economic} {temperature} {willingness} {ask_full}".replace("..", ".")
    
    # ABLATION 3: No Demographics (keep only GDP from economic)
    prompts['no_demo'] = f"In {country}, {gdp_only} {religion_only} {temperature} {willingness} {ask_full}".replace("..", ".")
    
    # ABLATION 4: No Climate
    prompts['no_climate'] = f"In {country}, {socio_full} {macro_economic} {willingness} {ask_full}".replace("..", ".")
    
    # ABLATION 5: No Own Willingness (CRITICAL - removes the "answer key")
    prompts['no_own_willingness'] = f"In {country}, {socio_full} {macro_economic} {temperature} {ask_full}".replace("..", ".")
    
    # ABLATION 6: Country Only
    ask_minimal = (
        f"In a nationally representative survey with a probability-based sample of approximately 1000 residents aged 15 and above in {country}, "
        f"respondents were asked how many respondents in {country} they think are willing to contribute at least 1% of their household income every month to fight global warming. "
        f"Based only on the country name, estimate what respondents in {country} on average thought about how many OTHER respondents in {country} are willing to contribute at least 1% of their household income every month to fight global warming. "
        "Respond with a single number between 0 and 100, with one decimal place."
    )
    prompts['country_only'] = f"In {country}, {ask_minimal}".replace("..", ".")
    
    # ================================================================
    # COUNTERFACTUALS
    # ================================================================
    
    # Determine if rich or poor (for GDP flip)
    gdp = row.get('gdp_capita_2021', 0)
    is_rich = gdp > 20000
    
    # CF1: GDP FLIP
    if is_rich:
        cf_gdp_str = "$2,000"
    else:
        cf_gdp_str = "$65,000"
    
    macro_economic_flipped = (
        f"GDP per capita (PPP, 2021) is {cf_gdp_str}, "
        f"the top 1% holds {fmt_pct(row.get('top1pct_income'))} of total income, "
        f"and {fmt_pct(row.get('top1pct_wealth'))} of total wealth. "
        f"The Human Development Index (2021) is {fmt_num(row.get('hdi_2021'), 3)}."
    )
    
    prompts['cf_gdp_flip'] = f"In {country}, {socio_full} {macro_economic_flipped} {temperature} {willingness} {ask_full}".replace("..", ".")
    
    # CF2: NAME-DATA MISMATCH
    all_countries_sorted = sorted(all_countries_df['countrynew'].unique())
    country_idx = all_countries_sorted.index(country) if country in all_countries_sorted else 0
    
    if is_rich:
        # Rich country data → Poor country name
        poor_countries = all_countries_df[all_countries_df['gdp_capita_2021'] < 5000]['countrynew'].tolist()
        if poor_countries:
            cf_name = sorted(poor_countries)[country_idx % len(poor_countries)]
        else:
            cf_name = "Chad"
    else:
        # Poor country data → Rich country name
        rich_countries = all_countries_df[all_countries_df['gdp_capita_2021'] > 40000]['countrynew'].tolist()
        if rich_countries:
            cf_name = sorted(rich_countries)[country_idx % len(rich_countries)]
        else:
            cf_name = "Norway"
    
    # Build prompt with mismatched name
    ask_mismatch = ask_full.replace(country, cf_name)
    prompts['cf_name_mismatch'] = f"In {cf_name}, {socio_full} {macro_economic} {temperature} {willingness} {ask_mismatch}".replace("..", ".")
    
    # CF3: OWN WILLINGNESS FLIP
    # Determine if high or low willingness country
    own_willingness = row.get('mean_own_willingness', 0)
    is_high_willingness = own_willingness > 50  # Countries with >50% willingness
    
    if is_high_willingness:
        # High willingness → flip to low (e.g., 15%)
        cf_own_main = "15.0%"
        cf_own_less = "10.0%"  # Also low
    else:
        # Low willingness → flip to high (e.g., 75%)
        cf_own_main = "75.0%"
        cf_own_less = "15.0%"  # Additional willing at lower amount
    
    willingness_flipped = (
        f"In this survey, {cf_own_main} of people said they are personally willing to contribute 1% of their income each month, "
        f"and an additional {cf_own_less} would contribute a smaller amount."
    )
    
    prompts['cf_willingness_flip'] = f"In {country}, {socio_full} {macro_economic} {temperature} {willingness_flipped} {ask_full}".replace("..", ".")
    
    return prompts

# ================================================================
# API Calling Functions
# ================================================================

def call_gpt(prompt, model="gpt-4o-mini", max_retries=3):
    """Call OpenAI API with tools disabled and JSON mode."""
    if not OPENAI_API_KEY:
        return None
    
    client = OpenAI(api_key=OPENAI_API_KEY)
    
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM_INSTRUCTION},
                    {"role": "user", "content": prompt}
                ],
                temperature=0,  # Deterministic output
                response_format={"type": "json_object"},  # JSON-only output
            )
            content = response.choices[0].message.content
            
            # Post-validation: check for forbidden strings
            if contains_forbidden_strings(content):
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                else:
                    return None
            
            return extract_number_0_100(content)
        except Exception as e:
            print(f"❌ GPT attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    return None

def call_claude(prompt, model="claude-3-5-haiku-20241022", max_retries=5):
    """Call Anthropic API with tools disabled."""
    if not CLAUDE_API_KEY:
        return None
    
    client = anthropic.Anthropic(api_key=CLAUDE_API_KEY)
    
    for attempt in range(max_retries):
        try:
            response = client.messages.create(
                model=model,
                max_tokens=100,
                temperature=0,  # Deterministic output
                system=SYSTEM_INSTRUCTION,
                messages=[
                    {"role": "user", "content": prompt}
                ],
                tools=[],  # No tools available
            )
            content = response.content[0].text
            
            # Post-validation: check for forbidden strings
            if contains_forbidden_strings(content):
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                else:
                    return None
            
            return extract_number_0_100(content)
        except anthropic.RateLimitError as e:
            # Rate limit - wait longer
            wait_time = (2 ** attempt) * 2
            print(f"⚠️  Claude rate limit, waiting {wait_time}s...")
            if attempt < max_retries - 1:
                time.sleep(wait_time)
            else:
                print(f"❌ Claude: Rate limit exceeded")
                return None
        except Exception as e:
            print(f"❌ Claude attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    return None

def call_gemini(prompt, model="gemini-1.5-flash", max_retries=3):
    """Call Google Gemini API."""
    if not GEMINI_API_KEY:
        return None
    
    genai.configure(api_key=GEMINI_API_KEY)
    model_obj = genai.GenerativeModel(model)
    
    full_prompt = f"{SYSTEM_INSTRUCTION}\n\n{prompt}"
    
    for attempt in range(max_retries):
        try:
            response = model_obj.generate_content(
                full_prompt,
                generation_config=genai.types.GenerationConfig(
                    temperature=0,
                )
            )
            content = response.text
            
            # Post-validation
            if contains_forbidden_strings(content):
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                else:
                    return None
            
            return extract_number_0_100(content)
        except Exception as e:
            print(f"❌ Gemini attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    return None

def call_llama(prompt, model="meta-llama/Llama-3.3-70B-Instruct-Turbo", max_retries=3):
    """Call Llama via Together API."""
    if not LLAMA_API_KEY:
        return None
    
    client = Together(api_key=LLAMA_API_KEY)
    
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM_INSTRUCTION},
                    {"role": "user", "content": prompt}
                ],
                temperature=0,
            )
            content = response.choices[0].message.content
            
            # Post-validation
            if contains_forbidden_strings(content):
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                else:
                    return None
            
            return extract_number_0_100(content)
        except Exception as e:
            print(f"❌ Llama attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    return None

# ================================================================
# Checkpoint Management
# ================================================================

def load_checkpoint():
    """Load checkpoint if it exists."""
    if Path(CHECKPOINT_FILE).exists():
        with open(CHECKPOINT_FILE, 'r') as f:
            return json.load(f)
    return {"completed": []}

def save_checkpoint(checkpoint):
    """Save checkpoint."""
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(checkpoint, f)

def is_completed(checkpoint, country, condition, model):
    """Check if a specific combination was completed."""
    key = f"{country}|{condition}|{model}"
    return key in checkpoint["completed"]

def mark_completed(checkpoint, country, condition, model):
    """Mark a combination as completed."""
    key = f"{country}|{condition}|{model}"
    if key not in checkpoint["completed"]:
        checkpoint["completed"].append(key)

# ================================================================
# Main Ablation Study
# ================================================================

def run_ablation_study():
    """Run the complete ablation study."""
    
    print("\n2. Loading checkpoint...")
    checkpoint = load_checkpoint()
    print(f"   ✓ {len(checkpoint['completed'])} tasks already completed")
    
    # Initialize or load results
    if Path(RAW_RESULTS_FILE).exists():
        results_df = pd.read_csv(RAW_RESULTS_FILE)
        print(f"   ✓ Loaded existing results: {len(results_df)} rows")
    else:
        results_df = pd.DataFrame()
        print("   ✓ Starting fresh results file")
    
    # Calculate total tasks
    total_tasks = len(df) * len(condition_order) * len([m for m in MODELS.keys() if os.getenv(f"{m.upper()}_API_KEY") or m == "gpt" and OPENAI_API_KEY or m == "claude" and CLAUDE_API_KEY or m == "gemini" and GEMINI_API_KEY or m == "llama" and LLAMA_API_KEY])
    completed_tasks = len(checkpoint['completed'])
    
    print(f"\n3. Running ablation study...")
    print(f"   Total tasks: {total_tasks}")
    print(f"   Remaining: {total_tasks - completed_tasks}")
    print(f"   Progress: {100 * completed_tasks / total_tasks:.1f}%\n")
    
    results = []
    start_time = time.time()
    
    for i, (idx, row) in enumerate(df.iterrows()):
        country = row['countrynew']
        
        print(f"\n[{i+1}/{len(df)}] Processing: {country}")
        
        # Build all prompts for this country
        all_prompts = build_ablated_prompts(row, df)
        
        # Test each condition
        for condition in condition_order:
            prompt = all_prompts[condition]
            
            # Test each model
            for model_name, model_id in MODELS.items():
                # Skip if already completed
                if is_completed(checkpoint, country, condition, model_name):
                    continue
                
                # Check if API key is available
                if model_name == "gpt" and not OPENAI_API_KEY:
                    continue
                if model_name == "claude" and not CLAUDE_API_KEY:
                    continue
                if model_name == "gemini" and not GEMINI_API_KEY:
                    continue
                if model_name == "llama" and not LLAMA_API_KEY:
                    continue
                
                print(f"  [{condition:20s}] [{model_name:7s}] ", end="", flush=True)
                
                # Call the appropriate API
                if model_name == "gpt":
                    prediction = call_gpt(prompt, model_id)
                elif model_name == "claude":
                    prediction = call_claude(prompt, model_id)
                elif model_name == "gemini":
                    prediction = call_gemini(prompt, model_id)
                elif model_name == "llama":
                    prediction = call_llama(prompt, model_id)
                else:
                    prediction = None
                
                if prediction is not None:
                    print(f"✓ {prediction:.1f}")
                else:
                    print(f"✗ Failed")
                
                # Save result
                results.append({
                    'country': country,
                    'condition': condition,
                    'model': model_name,
                    'prediction': prediction,
                    'actual_other_willingness': row.get('mean_other_willingness'),
                    'actual_own_willingness': row.get('mean_own_willingness'),
                })
                
                # Mark as completed
                mark_completed(checkpoint, country, condition, model_name)
                completed_tasks += 1
                
                # Save checkpoint every 10 tasks
                if completed_tasks % 10 == 0:
                    save_checkpoint(checkpoint)
                    
                    # Save results
                    if results:
                        new_results_df = pd.DataFrame(results)
                        if len(results_df) > 0:
                            results_df = pd.concat([results_df, new_results_df], ignore_index=True)
                        else:
                            results_df = new_results_df
                        results_df.to_csv(RAW_RESULTS_FILE, index=False)
                        results = []  # Clear buffer
                    
                    # Time estimate
                    elapsed = time.time() - start_time
                    rate = completed_tasks / elapsed
                    remaining = total_tasks - completed_tasks
                    eta_seconds = remaining / rate if rate > 0 else 0
                    eta_minutes = eta_seconds / 60
                    
                    print(f"\n  💾 Checkpoint saved | Progress: {100 * completed_tasks / total_tasks:.1f}% | ETA: {eta_minutes:.1f} min\n")
    
    # Final save
    if results:
        new_results_df = pd.DataFrame(results)
        if len(results_df) > 0:
            results_df = pd.concat([results_df, new_results_df], ignore_index=True)
        else:
            results_df = new_results_df
        results_df.to_csv(RAW_RESULTS_FILE, index=False)
    
    save_checkpoint(checkpoint)
    
    print("\n" + "="*80)
    print("ABLATION STUDY COMPLETE!")
    print("="*80)
    print(f"Results saved to: {RAW_RESULTS_FILE}")
    print(f"Total predictions: {len(results_df)}")
    print(f"Time elapsed: {(time.time() - start_time) / 60:.1f} minutes")
    
    return results_df

# ================================================================
# Run the study
# ================================================================

if __name__ == "__main__":
    try:
        results = run_ablation_study()
    except KeyboardInterrupt:
        print("\n\n⚠️  Interrupted by user")
        print("Progress has been saved. Run again to resume.")
    except Exception as e:
        print(f"\n\n❌ Error: {e}")
        print("Progress has been saved. Run again to resume.")
        raise

ABLATION STUDY - ORIGINAL STAGE 8 PROMPTS
Started: 2025-11-27 01:55:07

⏱️  [01:55:07] Session alive
✓ Keep-alive thread started (prevents timeout)


1. Loading data...
   ✓ Loaded 125 countries
   ✓ Temperature column: temp_mean_2010_2019
   ✓ Willingness_less column: mean_own_willigness_less

2. Loading checkpoint...
   ✓ 4980 tasks already completed
   ✓ Loaded existing results: 4990 rows

3. Running ablation study...
   Total tasks: 5000
   Remaining: 20
   Progress: 99.6%


[1/125] Processing: Afghanistan
  [full                ] [gpt    ] ✓ 60.0
  [full                ] [claude ] ✓ 42.5
  [full                ] [gemini ] ✓ 76.5
  [full                ] [llama  ] ✓ 58.2
  [no_econ             ] [gpt    ] ✓ 60.0
  [no_econ             ] [claude ] ✓ 45.6
  [no_econ             ] [gemini ] ✓ 68.5
  [no_econ             ] [llama  ] ✓ 58.2
  [no_religion         ] [gpt    ] ✓ 60.0
  [no_religion         ] [claude ] ✓ 42.5

  💾 Checkpoint saved | Progress: 99.8% | ETA: 0.0 min

  [no_re

In [1]:
"""
ABLATION STUDY - USING ORIGINAL STAGE 8 PROMPTS
================================================
Tests which features matter most for LLM predictions by systematically removing them.

This version uses the EXACT prompt format from the original study (Stage 8)
but applies the same ablation methodology from the previous ablation study.

FEATURES:
1. ✓ Checkpoint/resume system (can resume after failures)
2. ✓ Keep-alive thread for long runs (prevents timeout)
3. ✓ Retry logic for API calls with exponential backoff
4. ✓ All same ablation conditions tested
5. ✓ Counterfactual tests (GDP flip, name-data mismatch, willingness flip)
6. ✓ Progress tracking and time estimates
7. ✓ Forbidden string detection (prevents academic citations)
8. ✓ Can fill missing predictions later

USAGE:
1. First run: python ablation_study_original_prompts.py
2. If interrupted: Just run again - will resume from checkpoint
3. To fill missing: Use fill_missing_original.py (created separately)
"""

import os, re, time, json, threading
from datetime import datetime
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# API imports
from openai import OpenAI
import anthropic
import google.generativeai as genai
from together import Together

print("="*80)
print("ABLATION STUDY - ORIGINAL STAGE 8 PROMPTS")
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

# ================================================================
# Keep-alive thread (prevents timeout on long runs)
# ================================================================

def keep_alive():
    """Print periodic messages to keep session alive."""
    while True:
        if time.time() % 300 < 60:  # Every 5 minutes
            print(f"\n⏱️  [{datetime.now().strftime('%H:%M:%S')}] Session alive", flush=True)
        else:
            print(".", end="", flush=True)
        time.sleep(60)

keep_alive_thread = threading.Thread(target=keep_alive, daemon=True)
keep_alive_thread.start()
print("✓ Keep-alive thread started (prevents timeout)\n")

# ================================================================
# Configuration
# ================================================================

CHECKPOINT_FILE = "ablation_original_checkpoint.json"
RAW_RESULTS_FILE = "ablation_original_raw_results.csv"
DATA_FILE = "data_final.csv"

# API Configuration
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
CLAUDE_API_KEY = os.getenv("CLAUDE_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
LLAMA_API_KEY = os.getenv("LLAMA_API_KEY")

# Model configurations
MODELS = {
    "gpt": "gpt-4o-mini",
    "claude": "claude-3-5-haiku-20241022",
    "gemini": "gemini-2.5-flash",  # Google AI Studio: Gemini 2.5 Flash
    "llama": "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8"  # Together AI: Meta's Llama 4 Maverick
}

# System instruction to prevent data leakage and ensure lower-bound performance test
SYSTEM_INSTRUCTION = """You are a prediction assistant making estimates based ONLY on the information provided in this specific prompt.

CRITICAL INSTRUCTIONS:
1. Do NOT cite, reference, or mention ANY research papers, academic studies, surveys, or authors (including but not limited to André et al., Sparkman et al., Leviston et al., Lees et al., or any other researchers)
2. Do NOT use any memorized data, statistics, or percentages from your training about climate change opinions, pluralistic ignorance, or survey results
3. Treat this as a completely NOVEL scenario - ignore any similar studies you may have seen during training
4. Do NOT reference 'research shows', 'studies indicate', 'surveys have found', or similar phrases
5. Base your estimate ONLY on:
   - General reasoning about human psychology and behavior
   - The specific information provided in this prompt
   - First principles about how people form beliefs about others

Your task is to predict what percentage people THINK others believe (second-order belief), not what people actually believe (first-order belief). This is a prediction task requiring general reasoning, not recall of specific research findings.

Respond with ONLY a JSON object containing a single number between 0 and 100 with one decimal place: {"prediction": XX.X}

Do not include any explanation, reasoning, or text - only the JSON."""

# Condition order (INCLUDING NEW COUNTERFACTUAL)
condition_order = [
    'full',                 # Baseline - all features
    'no_econ',             # Remove economic indicators
    'no_religion',         # Remove religion
    'no_demo',             # Remove demographics
    'no_climate',          # Remove climate
    'no_own_willingness',  # Remove personal willingness (CRITICAL)
    'country_only',        # Only country name
    'cf_gdp_flip',         # Counterfactual: flip GDP
    'cf_name_mismatch',    # Counterfactual: mismatch name and data
    'cf_willingness_flip'  # NEW: Counterfactual: flip own willingness
]

# ================================================================
# Helper Functions
# ================================================================

def as_num(x, nd=1):
    """Convert to number with specified decimal places."""
    if pd.isna(x): 
        return None
    try:
        return round(float(x), nd)
    except:
        return None

def as_pct(x):
    """Return percentage (0-100) as float. Accept 0–1 or 0–100."""
    if pd.isna(x): 
        return None
    try:
        v = float(x)
    except:
        return None
    if 0 <= v <= 1:  # share format (0-1)
        return v * 100.0
    return v

def fmt_pct(x):
    """Format as percentage string."""
    v = as_pct(x)
    return f"{v:.1f}%" if v is not None else "N/A"

def fmt_num(x, nd=1):
    """Format as number string."""
    v = as_num(x, nd)
    return f"{v:.{nd}f}" if v is not None else "N/A"

def extract_number_0_100(text):
    """Extract a number between 0-100 from text or JSON."""
    if not isinstance(text, str):
        return None
    
    # Try parsing as JSON first
    try:
        data = json.loads(text)
        if isinstance(data, dict):
            # Look for common keys
            for key in ["prediction", "estimate", "value", "number", "percentage"]:
                if key in data:
                    val = float(data[key])
                    return max(0.0, min(100.0, val))
        elif isinstance(data, (int, float)):
            val = float(data)
            return max(0.0, min(100.0, val))
    except:
        pass
    
    # Fallback: regex extraction
    m = re.search(r"(\d+(?:\.\d+)?)", text)
    if not m:
        return None
    val = float(m.group(1))
    return max(0.0, min(100.0, val))

def contains_forbidden_strings(text):
    """Check if response contains forbidden strings that suggest academic citation."""
    if not isinstance(text, str):
        return False
    
    text_lower = text.lower()
    forbidden = [
        "andré", "andre",  # Author name variations
        "et al", "et. al", "et.al",  # Citation markers
        "doi", "http://", "https://",  # Links/DOIs
        "paper", "study", "research",  # Academic references (be careful with these)
        "published", "journal", "article"  # Publication terms
    ]
    
    return any(term in text_lower for term in forbidden)

# ================================================================
# Load Data
# ================================================================

print("\n1. Loading data...")
if not Path(DATA_FILE).exists():
    print(f"   ❌ Data file not found: {DATA_FILE}")
    print("   Please ensure data_final.csv is in the same directory")
    exit(1)

df = pd.read_csv(DATA_FILE)
print(f"   ✓ Loaded {len(df)} countries")

# Identify column names (handle spelling variations)
OWN_LESS_COL = None
for col_name in ["mean_own_willingness_less", "mean_own_willigness_less"]:
    if col_name in df.columns:
        OWN_LESS_COL = col_name
        break

TEMP_COL = None
for col_name in ["temp_mean", "temp_mean_2010_2019"]:
    if col_name in df.columns:
        TEMP_COL = col_name
        break

print(f"   ✓ Temperature column: {TEMP_COL}")
print(f"   ✓ Willingness_less column: {OWN_LESS_COL}")

# ================================================================
# Original Stage 8 Prompt Builder
# ================================================================

def build_prompt_stage8_original(row):
    """
    Build the ORIGINAL Stage 8 prompt exactly as in the original study.
    This is the baseline "full" condition.
    """
    country = row["countrynew"]
    
    socio_demographics = (
        f"The average age of respondents is {fmt_num(row.get('mean_age'), 1)} years, "
        f"{fmt_pct(row.get('mean_edu'))} of the people have completed a tertiary education, "
        f"and {fmt_pct(row.get('mean_religion'))} say religion is important in daily life."
    )
    
    macro_economic = (
        f"GDP per capita (PPP, 2021) is ${fmt_num(row.get('gdp_capita_2021'), 0)}, "
        f"the top 1% holds {fmt_pct(row.get('top1pct_income'))} of total income, "
        f"and {fmt_pct(row.get('top1pct_wealth'))} of total wealth. "
        f"The Human Development Index (2021) is {fmt_num(row.get('hdi_2021'), 3)}."
    )
    
    if TEMP_COL:
        temperature = f"The average temperature from 2010 to 2019 is {fmt_num(row.get(TEMP_COL), 1)}°C."
    else:
        temperature = "Temperature data are not available."
    
    own_main = fmt_pct(row.get("mean_own_willingness"))
    own_less = fmt_pct(row.get(OWN_LESS_COL)) if OWN_LESS_COL else "N/A"
    
    willingness = (
        f"In this survey, {own_main} of people said they are personally willing to contribute 1% of their income each month, "
        f"and an additional {own_less} would contribute a smaller amount."
    )
    
    ask = (
        f"In a nationally representative survey with a probability-based sample of approximately 1000 residents aged 15 and above in {country}, respondents were asked: "
        f"'Would you be willing to contribute 1% of your household income every month to fight global warming? This would mean that you would contribute 1 for every 100 of this income.' "
        f"Responses: Yes, No, (Don't Know), (Refused). Don't know and refused were coded as missing data. Respondents were then asked how many respondents in {country} they think are willing to contribute at least 1% of their household income every month to fight global warming. "
        f"Responses: between 0% and 100%, (Don't know), (Refused). Based on the country, socio-demographic, macro-economic indicators, temperature data, and the actual willingness data shown above, estimate what respondents in {country} on average thought about how many OTHER respondents in {country} are willing to contribute at least 1% of their household income every month to fight global warming. Note: You are estimating people's BELIEFS about others' willingness, not the actual willingness itself. "
        "Respond with a single number between 0 and 100, with one decimal place."
        
    )
    
    return f"In {country}, {socio_demographics} {macro_economic} {temperature} {willingness} {ask}".replace("..", ".")

# ================================================================
# Ablation Prompt Builders (using original Stage 8 format)
# ================================================================

def build_ablated_prompts(row, all_countries_df):
    """
    Build all ablation versions using the original Stage 8 format.
    Returns a dict with all conditions.
    """
    country = row["countrynew"]
    prompts = {}
    
    # Build reusable components
    socio_full = (
        f"The average age of respondents is {fmt_num(row.get('mean_age'), 1)} years, "
        f"{fmt_pct(row.get('mean_edu'))} of the people have completed a tertiary education, "
        f"and {fmt_pct(row.get('mean_religion'))} say religion is important in daily life."
    )
    
    socio_no_religion = (
        f"The average age of respondents is {fmt_num(row.get('mean_age'), 1)} years, "
        f"and {fmt_pct(row.get('mean_edu'))} of the people have completed a tertiary education."
    )
    
    macro_economic = (
        f"GDP per capita (PPP, 2021) is ${fmt_num(row.get('gdp_capita_2021'), 0)}, "
        f"the top 1% holds {fmt_pct(row.get('top1pct_income'))} of total income, "
        f"and {fmt_pct(row.get('top1pct_wealth'))} of total wealth. "
        f"The Human Development Index (2021) is {fmt_num(row.get('hdi_2021'), 3)}."
    )
    
    gdp_only = f"GDP per capita (PPP, 2021) is ${fmt_num(row.get('gdp_capita_2021'), 0)}."
    
    religion_only = f"{fmt_pct(row.get('mean_religion'))} say religion is important in daily life."
    
    if TEMP_COL:
        temperature = f"The average temperature from 2010 to 2019 is {fmt_num(row.get(TEMP_COL), 1)}°C."
    else:
        temperature = "Temperature data are not available."
    
    own_main = fmt_pct(row.get("mean_own_willingness"))
    own_less = fmt_pct(row.get(OWN_LESS_COL)) if OWN_LESS_COL else "N/A"
    
    willingness = (
        f"In this survey, {own_main} of people said they are personally willing to contribute 1% of their income each month, "
        f"and an additional {own_less} would contribute a smaller amount."
    )
    
    ask_full = (
        f"In a nationally representative survey with a probability-based sample of approximately 1000 residents aged 15 and above in {country}, respondents were asked: "
        f"'Would you be willing to contribute 1% of your household income every month to fight global warming? This would mean that you would contribute 1 for every 100 of this income.' "
        f"Responses: Yes, No, (Don't Know), (Refused). Don't know and refused were coded as missing data. Respondents were then asked how many respondents in {country} they think are willing to contribute at least 1% of their household income every month to fight global warming. "
        f"Responses: between 0% and 100%, (Don't know), (Refused). Based on the country, socio-demographic, macro-economic indicators, temperature data, and the actual willingness data shown above, estimate what respondents in {country} on average thought about how many OTHER respondents in {country} are willing to contribute at least 1% of their household income every month to fight global warming. Note: You are estimating people's BELIEFS about others' willingness, not the actual willingness itself. "
        "Respond with a single number between 0 and 100, with one decimal place."
    )
    
    # FULL PROMPT (baseline)
    prompts['full'] = build_prompt_stage8_original(row)
    
    # ABLATION 1: No Economic Indicators
    prompts['no_econ'] = f"In {country}, {socio_full} {temperature} {willingness} {ask_full}".replace("..", ".")
    
    # ABLATION 2: No Religion
    prompts['no_religion'] = f"In {country}, {socio_no_religion} {macro_economic} {temperature} {willingness} {ask_full}".replace("..", ".")
    
    # ABLATION 3: No Demographics (keep only GDP from economic)
    prompts['no_demo'] = f"In {country}, {gdp_only} {religion_only} {temperature} {willingness} {ask_full}".replace("..", ".")
    
    # ABLATION 4: No Climate
    prompts['no_climate'] = f"In {country}, {socio_full} {macro_economic} {willingness} {ask_full}".replace("..", ".")
    
    # ABLATION 5: No Own Willingness (CRITICAL - removes the "answer key")
    prompts['no_own_willingness'] = f"In {country}, {socio_full} {macro_economic} {temperature} {ask_full}".replace("..", ".")
    
    # ABLATION 6: Country Only
    ask_minimal = (
        f"In a nationally representative survey with a probability-based sample of approximately 1000 residents aged 15 and above in {country}, "
        f"respondents were asked how many respondents in {country} they think are willing to contribute at least 1% of their household income every month to fight global warming. "
        f"Based only on the country name, estimate what respondents in {country} on average thought about how many OTHER respondents in {country} are willing to contribute at least 1% of their household income every month to fight global warming. "
        "Respond with a single number between 0 and 100, with one decimal place."
    )
    prompts['country_only'] = f"In {country}, {ask_minimal}".replace("..", ".")
    
    # ================================================================
    # COUNTERFACTUALS
    # ================================================================
    
    # Determine if rich or poor (for GDP flip)
    gdp = row.get('gdp_capita_2021', 0)
    is_rich = gdp > 20000
    
    # CF1: GDP FLIP
    if is_rich:
        cf_gdp_str = "$2,000"
    else:
        cf_gdp_str = "$65,000"
    
    macro_economic_flipped = (
        f"GDP per capita (PPP, 2021) is {cf_gdp_str}, "
        f"the top 1% holds {fmt_pct(row.get('top1pct_income'))} of total income, "
        f"and {fmt_pct(row.get('top1pct_wealth'))} of total wealth. "
        f"The Human Development Index (2021) is {fmt_num(row.get('hdi_2021'), 3)}."
    )
    
    prompts['cf_gdp_flip'] = f"In {country}, {socio_full} {macro_economic_flipped} {temperature} {willingness} {ask_full}".replace("..", ".")
    
    # CF2: NAME-DATA MISMATCH
    all_countries_sorted = sorted(all_countries_df['countrynew'].unique())
    country_idx = all_countries_sorted.index(country) if country in all_countries_sorted else 0
    
    if is_rich:
        # Rich country data → Poor country name
        poor_countries = all_countries_df[all_countries_df['gdp_capita_2021'] < 5000]['countrynew'].tolist()
        if poor_countries:
            cf_name = sorted(poor_countries)[country_idx % len(poor_countries)]
        else:
            cf_name = "Chad"
    else:
        # Poor country data → Rich country name
        rich_countries = all_countries_df[all_countries_df['gdp_capita_2021'] > 40000]['countrynew'].tolist()
        if rich_countries:
            cf_name = sorted(rich_countries)[country_idx % len(rich_countries)]
        else:
            cf_name = "Norway"
    
    # Build prompt with mismatched name
    ask_mismatch = ask_full.replace(country, cf_name)
    prompts['cf_name_mismatch'] = f"In {cf_name}, {socio_full} {macro_economic} {temperature} {willingness} {ask_mismatch}".replace("..", ".")
    
    # CF3: OWN WILLINGNESS FLIP
    # Determine if high or low willingness country
    own_willingness = row.get('mean_own_willingness', 0)
    is_high_willingness = own_willingness > 50  # Countries with >50% willingness
    
    if is_high_willingness:
        # High willingness → flip to low (e.g., 15%)
        cf_own_main = "15.0%"
        cf_own_less = "10.0%"  # Also low
    else:
        # Low willingness → flip to high (e.g., 75%)
        cf_own_main = "75.0%"
        cf_own_less = "15.0%"  # Additional willing at lower amount
    
    willingness_flipped = (
        f"In this survey, {cf_own_main} of people said they are personally willing to contribute 1% of their income each month, "
        f"and an additional {cf_own_less} would contribute a smaller amount."
    )
    
    prompts['cf_willingness_flip'] = f"In {country}, {socio_full} {macro_economic} {temperature} {willingness_flipped} {ask_full}".replace("..", ".")
    
    return prompts

# ================================================================
# API Calling Functions
# ================================================================

def call_gpt(prompt, model="gpt-4o-mini", max_retries=3):
    """Call OpenAI API with tools disabled and JSON mode."""
    if not OPENAI_API_KEY:
        return None
    
    client = OpenAI(api_key=OPENAI_API_KEY)
    
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM_INSTRUCTION},
                    {"role": "user", "content": prompt}
                ],
                temperature=0,  # Deterministic output
                response_format={"type": "json_object"},  # JSON-only output
            )
            content = response.choices[0].message.content
            
            # Post-validation: check for forbidden strings
            if contains_forbidden_strings(content):
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                else:
                    return None
            
            return extract_number_0_100(content)
        except Exception as e:
            print(f"❌ GPT attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    return None

def call_claude(prompt, model="claude-3-5-haiku-20241022", max_retries=5):
    """Call Anthropic API with tools disabled."""
    if not CLAUDE_API_KEY:
        return None
    
    client = anthropic.Anthropic(api_key=CLAUDE_API_KEY)
    
    for attempt in range(max_retries):
        try:
            response = client.messages.create(
                model=model,
                max_tokens=100,
                temperature=0,  # Deterministic output
                system=SYSTEM_INSTRUCTION,
                messages=[
                    {"role": "user", "content": prompt}
                ],
                tools=[],  # No tools available
            )
            content = response.content[0].text
            
            # Post-validation: check for forbidden strings
            if contains_forbidden_strings(content):
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                else:
                    return None
            
            return extract_number_0_100(content)
        except anthropic.RateLimitError as e:
            # Rate limit - wait longer
            wait_time = (2 ** attempt) * 2
            print(f"⚠️  Claude rate limit, waiting {wait_time}s...")
            if attempt < max_retries - 1:
                time.sleep(wait_time)
            else:
                print(f"❌ Claude: Rate limit exceeded")
                return None
        except Exception as e:
            print(f"❌ Claude attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    return None

def call_gemini(prompt, model="gemini-1.5-flash", max_retries=3):
    """Call Google Gemini API."""
    if not GEMINI_API_KEY:
        return None
    
    genai.configure(api_key=GEMINI_API_KEY)
    model_obj = genai.GenerativeModel(model)
    
    full_prompt = f"{SYSTEM_INSTRUCTION}\n\n{prompt}"
    
    for attempt in range(max_retries):
        try:
            response = model_obj.generate_content(
                full_prompt,
                generation_config=genai.types.GenerationConfig(
                    temperature=0,
                )
            )
            content = response.text
            
            # Post-validation
            if contains_forbidden_strings(content):
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                else:
                    return None
            
            return extract_number_0_100(content)
        except Exception as e:
            print(f"❌ Gemini attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    return None

def call_llama(prompt, model="meta-llama/Llama-3.3-70B-Instruct-Turbo", max_retries=3):
    """Call Llama via Together API."""
    if not LLAMA_API_KEY:
        return None
    
    client = Together(api_key=LLAMA_API_KEY)
    
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM_INSTRUCTION},
                    {"role": "user", "content": prompt}
                ],
                temperature=0,
            )
            content = response.choices[0].message.content
            
            # Post-validation
            if contains_forbidden_strings(content):
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                else:
                    return None
            
            return extract_number_0_100(content)
        except Exception as e:
            print(f"❌ Llama attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    return None

# ================================================================
# Checkpoint Management
# ================================================================

def load_checkpoint():
    """Load checkpoint if it exists."""
    if Path(CHECKPOINT_FILE).exists():
        with open(CHECKPOINT_FILE, 'r') as f:
            return json.load(f)
    return {"completed": []}

def save_checkpoint(checkpoint):
    """Save checkpoint."""
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(checkpoint, f)

def is_completed(checkpoint, country, condition, model):
    """Check if a specific combination was completed."""
    key = f"{country}|{condition}|{model}"
    return key in checkpoint["completed"]

def mark_completed(checkpoint, country, condition, model):
    """Mark a combination as completed."""
    key = f"{country}|{condition}|{model}"
    if key not in checkpoint["completed"]:
        checkpoint["completed"].append(key)

# ================================================================
# Main Ablation Study
# ================================================================

def run_ablation_study():
    """Run the complete ablation study."""
    
    print("\n2. Loading checkpoint...")
    checkpoint = load_checkpoint()
    print(f"   ✓ {len(checkpoint['completed'])} tasks already completed")
    
    # Initialize or load results
    if Path(RAW_RESULTS_FILE).exists():
        results_df = pd.read_csv(RAW_RESULTS_FILE)
        print(f"   ✓ Loaded existing results: {len(results_df)} rows")
    else:
        results_df = pd.DataFrame()
        print("   ✓ Starting fresh results file")
    
    # Calculate total tasks
    total_tasks = len(df) * len(condition_order) * len([m for m in MODELS.keys() if os.getenv(f"{m.upper()}_API_KEY") or m == "gpt" and OPENAI_API_KEY or m == "claude" and CLAUDE_API_KEY or m == "gemini" and GEMINI_API_KEY or m == "llama" and LLAMA_API_KEY])
    completed_tasks = len(checkpoint['completed'])
    
    print(f"\n3. Running ablation study...")
    print(f"   Total tasks: {total_tasks}")
    print(f"   Remaining: {total_tasks - completed_tasks}")
    print(f"   Progress: {100 * completed_tasks / total_tasks:.1f}%\n")
    
    results = []
    start_time = time.time()
    
    for i, (idx, row) in enumerate(df.iterrows()):
        country = row['countrynew']
        
        print(f"\n[{i+1}/{len(df)}] Processing: {country}")
        
        # Build all prompts for this country
        all_prompts = build_ablated_prompts(row, df)
        
        # Test each condition
        for condition in condition_order:
            prompt = all_prompts[condition]
            
            # Test each model
            for model_name, model_id in MODELS.items():
                # Skip if already completed
                if is_completed(checkpoint, country, condition, model_name):
                    continue
                
                # Check if API key is available
                if model_name == "gpt" and not OPENAI_API_KEY:
                    continue
                if model_name == "claude" and not CLAUDE_API_KEY:
                    continue
                if model_name == "gemini" and not GEMINI_API_KEY:
                    continue
                if model_name == "llama" and not LLAMA_API_KEY:
                    continue
                
                print(f"  [{condition:20s}] [{model_name:7s}] ", end="", flush=True)
                
                # Call the appropriate API
                if model_name == "gpt":
                    prediction = call_gpt(prompt, model_id)
                elif model_name == "claude":
                    prediction = call_claude(prompt, model_id)
                elif model_name == "gemini":
                    prediction = call_gemini(prompt, model_id)
                elif model_name == "llama":
                    prediction = call_llama(prompt, model_id)
                else:
                    prediction = None
                
                if prediction is not None:
                    print(f"✓ {prediction:.1f}")
                else:
                    print(f"✗ Failed")
                
                # Save result
                results.append({
                    'country': country,
                    'condition': condition,
                    'model': model_name,
                    'prediction': prediction,
                    'actual_other_willingness': row.get('mean_other_willingness'),
                    'actual_own_willingness': row.get('mean_own_willingness'),
                })
                
                # Mark as completed
                mark_completed(checkpoint, country, condition, model_name)
                completed_tasks += 1
                
                # Save checkpoint every 10 tasks
                if completed_tasks % 10 == 0:
                    save_checkpoint(checkpoint)
                    
                    # Save results
                    if results:
                        new_results_df = pd.DataFrame(results)
                        if len(results_df) > 0:
                            results_df = pd.concat([results_df, new_results_df], ignore_index=True)
                        else:
                            results_df = new_results_df
                        results_df.to_csv(RAW_RESULTS_FILE, index=False)
                        results = []  # Clear buffer
                    
                    # Time estimate
                    elapsed = time.time() - start_time
                    rate = completed_tasks / elapsed
                    remaining = total_tasks - completed_tasks
                    eta_seconds = remaining / rate if rate > 0 else 0
                    eta_minutes = eta_seconds / 60
                    
                    print(f"\n  💾 Checkpoint saved | Progress: {100 * completed_tasks / total_tasks:.1f}% | ETA: {eta_minutes:.1f} min\n")
    
    # Final save
    if results:
        new_results_df = pd.DataFrame(results)
        if len(results_df) > 0:
            results_df = pd.concat([results_df, new_results_df], ignore_index=True)
        else:
            results_df = new_results_df
        results_df.to_csv(RAW_RESULTS_FILE, index=False)
    
    save_checkpoint(checkpoint)
    
    print("\n" + "="*80)
    print("ABLATION STUDY COMPLETE!")
    print("="*80)
    print(f"Results saved to: {RAW_RESULTS_FILE}")
    print(f"Total predictions: {len(results_df)}")
    print(f"Time elapsed: {(time.time() - start_time) / 60:.1f} minutes")
    
    return results_df

# ================================================================
# Run the study
# ================================================================

if __name__ == "__main__":
    try:
        results = run_ablation_study()
    except KeyboardInterrupt:
        print("\n\n⚠️  Interrupted by user")
        print("Progress has been saved. Run again to resume.")
    except Exception as e:
        print(f"\n\n❌ Error: {e}")
        print("Progress has been saved. Run again to resume.")
        raise

ABLATION STUDY - ORIGINAL STAGE 8 PROMPTS
Started: 2025-11-27 02:29:26
.✓ Keep-alive thread started (prevents timeout)


1. Loading data...
   ✓ Loaded 125 countries
   ✓ Temperature column: temp_mean_2010_2019
   ✓ Willingness_less column: mean_own_willigness_less

2. Loading checkpoint...
   ✓ 5000 tasks already completed
   ✓ Loaded existing results: 5010 rows

3. Running ablation study...
   Total tasks: 5000
   Remaining: 0
   Progress: 100.0%


[1/125] Processing: Afghanistan

[2/125] Processing: Albania

[3/125] Processing: Algeria

[4/125] Processing: Argentina

[5/125] Processing: Armenia

[6/125] Processing: Australia

[7/125] Processing: Austria

[8/125] Processing: Bangladesh

[9/125] Processing: Belgium

[10/125] Processing: Benin

[11/125] Processing: Bolivia

[12/125] Processing: Bosnia Herzegovina

[13/125] Processing: Botswana

[14/125] Processing: Brazil

[15/125] Processing: Bulgaria

[16/125] Processing: Burkina Faso

[17/125] Processing: Cambodia

[18/125] Processi

In [4]:
# ========================================================================
# FINAL DEDUPLICATION SCRIPT — produces exactly 5000 valid observations
# ========================================================================

import pandas as pd, shutil

file_original = "ablation_original_raw_results.csv"
file_backup   = "ablation_original_raw_results_backup_full.csv"

# ----------------------------------
# 1) Load & backup original file
# ----------------------------------
df = pd.read_csv(file_original)
shutil.copy(file_original, file_backup)   # safety copy

print(f"\nLoaded original file: {file_original}")
print(f"Original row count = {len(df)}")

# ----------------------------------
# 2) Remove exact row duplicates
# ----------------------------------
df_exact_clean = df.drop_duplicates()
print(f"After removing exact duplicates = {len(df_exact_clean)}")

# ----------------------------------
# 3) Identify non-identical duplicates by key
# ----------------------------------
dup_keys = (df_exact_clean.groupby(["country","model","condition"])
                         .size().reset_index(name="n"))
logical_dupes = dup_keys[dup_keys.n > 1]

print("\n🔎 Logical duplicates detected (same key, different values):")
print(logical_dupes)

# expected total extra rows = number over 5000
extra = len(df_exact_clean) - 5000
print(f"\nExtra remaining rows beyond expected = {extra}")

# ----------------------------------
# 4) Remove logical duplicates (keeping first instance)
# ----------------------------------
df_final = df_exact_clean.drop_duplicates(
    subset=["country","model","condition"],
    keep="first"                # keep first and discard the rest
)

df_final.to_csv(file_original, index=False)

print("\n==============================================")
print(f"FINAL dataset saved at: {file_original}  ✔")
print(f"Backup saved at        : {file_backup}")
print(f"Final row count        : {len(df_final)}")
print("==============================================\n")

if len(df_final) == 5000:
    print("🎉 SUCCESS — Dataset perfectly cleaned to 5000 rows.\n")
else:
    print("⚠ Unexpected row count — investigate further.\n")



Loaded original file: ablation_original_raw_results.csv
Original row count = 5003
After removing exact duplicates = 5003

🔎 Logical duplicates detected (same key, different values):
        country   model   condition  n
16  Afghanistan  gemini     no_demo  2
35  Afghanistan   llama  no_climate  2
36  Afghanistan   llama     no_demo  2

Extra remaining rows beyond expected = 3

FINAL dataset saved at: ablation_original_raw_results.csv  ✔
Backup saved at        : ablation_original_raw_results_backup_full.csv
Final row count        : 5000

🎉 SUCCESS — Dataset perfectly cleaned to 5000 rows.



<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=feb9f195-de2a-416f-b8f1-09efca4e954f' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>